#### Goal
- To Achive Bank code for party for now, in future other identifiers will be added

#### author
- Abhishek.R.Jaiswal@rabobank.com

##### flow of logic
- Read Data from GCDS and identify the bank codes

##### Version & Changes
|     Developer |Date	   | PBI/Bug No |	Changes done |
|----------|----------|----------|----------|
| Abhishek Jaiswal	 |13-Oct-2025 |13890287 |First release
| Mahalakshmi V | 20-Nov-2025 | 14165735 | Made notebook compatible for Historical Load 


#####Read Files from GDP

In [0]:
import os
app_reg_app_id = os.environ['APP_REG_APP_ID']
ReadStorage = os.environ['GDP_STORAGE_NAME']
TenantId = os.environ['TENANT_ID']
GIC_ReadStorage = os.environ['GDP_SA_STORAGE_NAME']
environment=os.environ['ENV']
radar_datamodel_version_number=3
service_credential = dbutils.secrets.get(scope="connectedsecrets", key = f"appreg-{app_reg_app_id}")
SARADAR = "saradar" + environment

In [0]:
from RadarUtils import *

In [0]:
dbutils.widgets.text("Load_Date", "")
dbutils.widgets.text("RunType", "daily")  # default is daily

Load_Date = dbutils.widgets.get("Load_Date")
RunType = dbutils.widgets.get("RunType").lower()
if Load_Date:
    RunType="historical"
    
if not Load_Date:
    Load_Date = datetime.today().strftime('%Y%m%d')

print(f"Running for Load Date: {Load_Date}")

In [0]:
party_dataobject='Party_RadarKeyStore'

In [0]:
authenticate_storage_account(ReadStorage)
#authenticate_storage_account(GIC_ReadStorage)
#authenticate_storage_account(SARADAR)

In [0]:
# Derive the date for which data has to be processed from GDP
load_dts = f"EDL_LOAD_DTS={Load_Date}*"
print (load_dts)

In [0]:
# print list of strings for loading spark dfs from GDP
gcds_dataobjects_list = [
'client_KeyStoreKey'
]

# Create TempView for each loading table
for dataobject in gcds_dataobjects_list:
    Read_GDP_Defined_DataObjects(Source='GCDS', Dataobject=dataobject,Load_Date=Load_Date)

In [0]:
#df_Party_SystemIdentifier = spark.read.parquet(
#    f"abfss://radardatamodel@{SARADAR}.dfs.core.windows.net/Party_SystemIdentifier/3/data/{load_dts}/*.parquet"
#)

#####GCDS - Data Preparation

In [0]:
%sql
Create or replace temporary view GCDS_KeyStore As
select distinct
KeyStore_value
,KeyStore_type
,status
,GCID
from client_KeyStoreKey


In [0]:
%sql
Create or replace temporary view GCDS_RADAR_KeyStore As
SELECT DISTINCT
CONCAT('GCDS_',gcid) AS PartyIdentifier
,KeyStore_type as KeyStore_Type
,KeyStore_value as KeyStore_Value
,status as Key_Status
FROM GCDS_KeyStore

In [0]:
df_RADAR_KeyStore=spark.table('GCDS_RADAR_KeyStore')

In [0]:
#df_party_Selection = add_party_identifier(df_party_Selection, df_Party_SystemIdentifier)

In [0]:
if RunType == "historical":
    save_to_saradar_storage_account(df_RADAR_KeyStore, party_dataobject, radar_datamodel_version_number, environment,Load_Date)
else:
    save_to_saradar_storage_account(df_RADAR_KeyStore, party_dataobject, radar_datamodel_version_number, environment)